# Machine Learning Algorithms from Scratch

**Module 02: Machine Learning**  
**Objective**: Implement core ML algorithms from first principles

Building ML algorithms from scratch helps you:
- Understand the mathematical foundations
- Debug issues in production systems
- Optimize for specific use cases
- Answer interview questions with depth

## What You'll Learn
1. Linear & Logistic Regression
2. K-Nearest Neighbors (KNN)
3. Decision Trees
4. Random Forests
5. Gradient Boosting
6. K-Means Clustering
7. PCA (Principal Component Analysis)
8. Support Vector Machines (SVM)

## 1. Linear Regression

**Problem**: Find the best-fit line $y = wx + b$

**Loss**: Mean Squared Error (MSE)

$$\mathcal{L}(w, b) = \frac{1}{N}\sum_{i=1}^{N}(y_i - (wx_i + b))^2$$

**Solution**: Gradient Descent

$$w \leftarrow w - \alpha \frac{\partial \mathcal{L}}{\partial w}$$
$$b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

np.random.seed(42)

In [ ]:
class LinearRegression:
    """Linear Regression with Gradient Descent."""
    
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []
    
    def fit(self, X, y):
        """
        Train the model.
        
        Args:
            X: (n_samples, n_features)
            y: (n_samples,)
        """
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.n_iters):
            # Forward pass
            y_pred = X @ self.weights + self.bias
            
            # Compute loss
            loss = np.mean((y - y_pred) ** 2)
            self.losses.append(loss)
            
            # Compute gradients
            dw = -(2/n_samples) * (X.T @ (y - y_pred))
            db = -(2/n_samples) * np.sum(y - y_pred)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
    
    def predict(self, X):
        """Make predictions."""
        return X @ self.weights + self.bias

# Test on synthetic data
X, y = make_regression(n_samples=1000, n_features=1, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression(lr=0.01, n_iters=1000)
model.fit(X_train, y_train)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluation
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
test_r2 = r2_score(y_test, y_pred_test)

print(f"Linear Regression Results:")
print(f"Train MSE: {train_mse:.2f}")
print(f"Test MSE: {test_mse:.2f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Learned weight: {model.weights[0]:.4f}, bias: {model.bias:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(model.losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Predictions
ax2.scatter(X_test, y_test, alpha=0.5, label='True')
ax2.scatter(X_test, y_pred_test, alpha=0.5, label='Predicted')
ax2.plot(X_test, y_pred_test, color='red', linewidth=2)
ax2.set_xlabel('X')
ax2.set_ylabel('y')
ax2.set_title('Test Set Predictions')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Logistic Regression

**Problem**: Binary classification

**Model**: 
$$\hat{y} = \sigma(wx + b) = \frac{1}{1 + e^{-(wx + b)}}$$

**Loss**: Binary Cross-Entropy
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

In [ ]:
class LogisticRegression:
    """Logistic Regression for binary classification."""
    
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []
    
    def _sigmoid(self, z):
        """Sigmoid activation."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Clip for numerical stability
    
    def fit(self, X, y):
        """Train the model."""
        n_samples, n_features = X.shape
        
        # Initialize
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.n_iters):
            # Forward
            linear_output = X @ self.weights + self.bias
            y_pred = self._sigmoid(linear_output)
            
            # Loss (binary cross-entropy)
            loss = -np.mean(y * np.log(y_pred + 1e-8) + (1 - y) * np.log(1 - y_pred + 1e-8))
            self.losses.append(loss)
            
            # Gradients
            dw = (1/n_samples) * (X.T @ (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
    
    def predict_proba(self, X):
        """Predict probabilities."""
        linear_output = X @ self.weights + self.bias
        return self._sigmoid(linear_output)
    
    def predict(self, X, threshold=0.5):
        """Predict class labels."""
        return (self.predict_proba(X) >= threshold).astype(int)

# Test
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                           n_informative=2, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
model = LogisticRegression(lr=0.1, n_iters=1000)
model.fit(X_train, y_train)

# Predict
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\nLogistic Regression Results:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Plot decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax1.plot(model.losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Binary Cross-Entropy Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Decision boundary
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), 
                     np.linspace(y_min, y_max, 100))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.3, levels=1)
ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k')
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title(f'Decision Boundary (Acc={test_acc:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. K-Nearest Neighbors (KNN)

**Algorithm**: For a test point, find k nearest neighbors and vote (classification) or average (regression).

**Distance metric**: Euclidean distance

$$d(x_i, x_j) = \sqrt{\sum_{k=1}^{d}(x_{ik} - x_{jk})^2}$$

In [ ]:
class KNNClassifier:
    """K-Nearest Neighbors Classifier."""
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """Store training data (lazy learning)."""
        self.X_train = X
        self.y_train = y
    
    def _euclidean_distance(self, x1, x2):
        """Compute Euclidean distance between two points."""
        return np.sqrt(np.sum((x1 - x2) ** 2))
    
    def predict(self, X):
        """Predict labels for test data."""
        predictions = []
        for x in X:
            # Compute distances to all training points
            distances = [self._euclidean_distance(x, x_train) for x_train in self.X_train]
            
            # Get k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_nearest_labels = self.y_train[k_indices]
            
            # Majority vote
            most_common = np.bincount(k_nearest_labels).argmax()
            predictions.append(most_common)
        
        return np.array(predictions)

# Vectorized version (much faster)
class KNNClassifierFast:
    """Vectorized KNN Classifier."""
    
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def predict(self, X):
        """Vectorized prediction."""
        # Compute pairwise distances: (n_test, n_train)
        # ||x - y||^2 = ||x||^2 + ||y||^2 - 2<x,y>
        X_sq = np.sum(X**2, axis=1, keepdims=True)
        X_train_sq = np.sum(self.X_train**2, axis=1, keepdims=True)
        cross_term = X @ self.X_train.T
        
        distances = np.sqrt(X_sq + X_train_sq.T - 2 * cross_term)
        
        # Get k nearest for each test point
        k_indices = np.argsort(distances, axis=1)[:, :self.k]
        k_nearest_labels = self.y_train[k_indices]
        
        # Majority vote for each test point
        predictions = np.array([np.bincount(labels).argmax() for labels in k_nearest_labels])
        
        return predictions

# Test
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0,
                           n_informative=2, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train and evaluate different k values
k_values = [1, 3, 5, 10, 20]
accuracies = []

for k in k_values:
    knn = KNNClassifierFast(k=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f"k={k:2d}: Test Accuracy = {acc:.4f}")

# Plot
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(k_values, accuracies, marker='o')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Test Accuracy')
plt.title('KNN: Effect of k')
plt.grid(True, alpha=0.3)

# Decision boundary for best k
best_k = k_values[np.argmax(accuracies)]
knn = KNNClassifierFast(k=best_k)
knn.fit(X_train, y_train)

plt.subplot(1, 2, 2)
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))
Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'KNN Decision Boundary (k={best_k})')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest k: {best_k} with accuracy: {max(accuracies):.4f}")

## 4. Decision Trees

**Algorithm**: Recursively split data based on feature that maximizes information gain.

**Gini Impurity**:
$$\text{Gini}(S) = 1 - \sum_{i=1}^{C} p_i^2$$

**Information Gain**:
$$\text{IG}(S, A) = \text{Gini}(S) - \sum_{v \in A}\frac{|S_v|}{|S|}\text{Gini}(S_v)$$

In [ ]:
class Node:
    """Decision tree node."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      # Feature index to split on
        self.threshold = threshold  # Threshold value
        self.left = left           # Left child
        self.right = right         # Right child
        self.value = value         # Leaf node value (class label)

class DecisionTreeClassifier:
    """Decision Tree for classification."""
    
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
    
    def _gini(self, y):
        """Compute Gini impurity."""
        proportions = np.bincount(y) / len(y)
        return 1 - np.sum(proportions ** 2)
    
    def _information_gain(self, y_parent, y_left, y_right):
        """Compute information gain."""
        weight_left = len(y_left) / len(y_parent)
        weight_right = len(y_right) / len(y_parent)
        
        gain = self._gini(y_parent) - (weight_left * self._gini(y_left) + 
                                       weight_right * self._gini(y_right))
        return gain
    
    def _best_split(self, X, y):
        """Find best feature and threshold to split on."""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            
            for threshold in thresholds:
                # Split
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue
                
                # Information gain
                gain = self._information_gain(y, y[left_mask], y[right_mask])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build decision tree."""
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # Stopping criteria
        if (depth >= self.max_depth or 
            n_labels == 1 or 
            n_samples < self.min_samples_split):
            leaf_value = np.bincount(y).argmax()
            return Node(value=leaf_value)
        
        # Find best split
        best_feature, best_threshold = self._best_split(X, y)
        
        if best_feature is None:
            leaf_value = np.bincount(y).argmax()
            return Node(value=leaf_value)
        
        # Split
        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask
        
        # Recursively build subtrees
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return Node(feature=best_feature, threshold=best_threshold, left=left, right=right)
    
    def fit(self, X, y):
        """Build decision tree."""
        self.root = self._build_tree(X, y)
    
    def _traverse_tree(self, x, node):
        """Traverse tree for single sample."""
        if node.value is not None:
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)
    
    def predict(self, X):
        """Predict labels."""
        return np.array([self._traverse_tree(x, self.root) for x in X])

# Test
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0,
                           n_informative=2, n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train
tree = DecisionTreeClassifier(max_depth=5, min_samples_split=5)
tree.fit(X_train, y_train)

# Predict
y_pred_train = tree.predict(X_train)
y_pred_test = tree.predict(X_test)

# Evaluate
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\nDecision Tree Results:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Plot decision boundary
plt.figure(figsize=(8, 6))
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))
Z = tree.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'Decision Tree Boundary (Acc={test_acc:.3f})')
plt.grid(True, alpha=0.3)
plt.show()

## 5. K-Means Clustering

**Algorithm**: Iteratively assign points to nearest centroid and update centroids.

**Steps**:
1. Initialize k centroids randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat until convergence

In [ ]:
class KMeans:
    """K-Means Clustering."""
    
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters
        self.centroids = None
        self.labels = None
        self.inertia_history = []
    
    def fit(self, X):
        """Fit K-Means."""
        n_samples, n_features = X.shape
        
        # Initialize centroids randomly (K-means++)
        self.centroids = self._kmeans_plusplus_init(X)
        
        for i in range(self.max_iters):
            # Assign points to nearest centroid
            labels = self._assign_clusters(X)
            
            # Store old centroids
            old_centroids = self.centroids.copy()
            
            # Update centroids
            for k in range(self.k):
                if np.sum(labels == k) > 0:
                    self.centroids[k] = X[labels == k].mean(axis=0)
            
            # Compute inertia (sum of squared distances)
            inertia = self._compute_inertia(X, labels)
            self.inertia_history.append(inertia)
            
            # Check convergence
            if np.allclose(old_centroids, self.centroids):
                print(f"Converged after {i+1} iterations")
                break
        
        self.labels = labels
        return self
    
    def _kmeans_plusplus_init(self, X):
        """K-means++ initialization."""
        centroids = []
        
        # Choose first centroid randomly
        centroids.append(X[np.random.choice(len(X))])
        
        # Choose remaining centroids
        for _ in range(1, self.k):
            # Compute distances to nearest centroid
            distances = np.array([min([np.linalg.norm(x - c)**2 for c in centroids]) for x in X])
            
            # Choose next centroid with probability proportional to distance^2
            probs = distances / distances.sum()
            cumprobs = np.cumsum(probs)
            r = np.random.rand()
            
            for j, p in enumerate(cumprobs):
                if r < p:
                    centroids.append(X[j])
                    break
        
        return np.array(centroids)
    
    def _assign_clusters(self, X):
        """Assign points to nearest centroid."""
        distances = np.array([[np.linalg.norm(x - c) for c in self.centroids] for x in X])
        return np.argmin(distances, axis=1)
    
    def _compute_inertia(self, X, labels):
        """Compute sum of squared distances to centroids."""
        inertia = 0
        for k in range(self.k):
            cluster_points = X[labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - self.centroids[k])**2)
        return inertia
    
    def predict(self, X):
        """Assign new points to clusters."""
        return self._assign_clusters(X)

# Test
X, y_true = make_blobs(n_samples=500, n_features=2, centers=4, random_state=42)

# Fit K-Means
kmeans = KMeans(k=4, max_iters=100)
kmeans.fit(X)

print(f"\nK-Means Results:")
print(f"Final inertia: {kmeans.inertia_history[-1]:.2f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Clusters
ax1.scatter(X[:, 0], X[:, 1], c=kmeans.labels, cmap='viridis', alpha=0.6)
ax1.scatter(kmeans. centroids[:, 0], kmeans.centroids[:, 1], 
           marker='X', s=300, c='red', edgecolors='black', linewidths=2)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('K-Means Clustering')
ax1.grid(True, alpha=0.3)

# Inertia curve
ax2.plot(kmeans.inertia_history)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Inertia')
ax2.set_title('Convergence')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Elbow method
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans_temp = KMeans(k=k, max_iters=100)
    kmeans_temp.fit(X)
    inertias.append(kmeans_temp.inertia_history[-1])

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nOptimal k is where the elbow appears (around k=4)")

## 6. PCA (Principal Component Analysis)

**Goal**: Find directions of maximum variance in data.

**Steps**:
1. Center the data: $X' = X - \mu$
2. Compute covariance matrix: $\Sigma = \frac{1}{n}X'^TX'$
3. Eigendecomposition: $\Sigma = V\Lambda V^T$
4. Project onto top k eigenvectors: $Z = X'V_k$

In [ ]:
class PCA:
    """Principal Component Analysis."""
    
    def __init__(self, n_components=2):
        self.n_components = n_components
        self.mean = None
        self.components = None
        self.explained_variance = None
    
    def fit(self, X):
        """Fit PCA."""
        # Center data
        self.mean = X.mean(axis=0)
        X_centered = X - self.mean
        
        # Covariance matrix
        cov = np.cov(X_centered.T)
        
        # Eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        
        # Sort by eigenvalues (descending)
        idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        
        # Store first n_components
        self.components = eigenvectors[:, :self.n_components]
        self.explained_variance = eigenvalues[:self.n_components]
        
        return self
    
    def transform(self, X):
        """Project data onto principal components."""
        X_centered = X - self.mean
        return X_centered @ self.components
    
    def fit_transform(self, X):
        """Fit and transform."""
        self.fit(X)
        return self.transform(X)
    
    def inverse_transform(self, X_transformed):
        """Reconstruct original data."""
        return X_transformed @ self.components.T + self.mean

# Test on high-dimensional data
X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                           n_redundant=10, random_state=42)

# Fit PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"\nPCA Results:")
print(f"Original shape: {X.shape}")
print(f"Transformed shape: {X_pca.shape}")
print(f"Explained variance: {pca.explained_variance}")
print(f"Explained variance ratio: {pca.explained_variance / pca.explained_variance.sum()}")
print(f"Total explained: {(pca.explained_variance / np.var(X, axis=0).sum()).sum():.2%}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 2D projection
ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.6)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('PCA Projection (2D)')
ax1.grid(True, alpha=0.3)

# Scree plot
pca_full = PCA(n_components=min(20, X.shape[1]))
pca_full.fit(X)
explained_var_ratio = pca_full.explained_variance / pca_full.explained_variance.sum()

ax2.bar(range(1, len(explained_var_ratio)+1), explained_var_ratio)
ax2.plot(range(1, len(explained_var_ratio)+1), np.cumsum(explained_var_ratio), 
         'ro-', label='Cumulative')
ax2.set_xlabel('Principal Component')
ax2.set_ylabel('Explained Variance Ratio')
ax2.set_title('Scree Plot')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

You've implemented from scratch:
- ✅ Linear Regression (gradient descent)
- ✅ Logistic Regression (binary classification)
- ✅ K-Nearest Neighbors (lazy learning)
- ✅ Decision Trees (recursive splitting)
- ✅ K-Means Clustering (unsupervised)
- ✅ PCA (dimensionality reduction)

**Key Takeaways**:
1. Most ML algorithms optimize a loss function
2. Gradient descent is ubiquitous
3. Trade-off: model complexity vs generalization
4. Feature engineering matters (GIGO: Garbage In, Garbage Out)

**Next Notebook**: Advanced ML with scikit-learn (Random Forests, Gradient Boosting, SVMs)